# Track T4: Multi-Layer Perceptron (MLP) Modeling, Tuning & Explainability
**Student Owner:** B2 (Evaluation Architect & Neural Network Specialist)

This notebook covers:
1. **Stage 9 Supervised Modeling:** Baseline `mlp_regressor` and `mlp_classifier`.
2. **Architectural Comparison:** Method 1 (Dedicated Softmax Classifier) vs. Method 2 (Thresholded Regressor).
3. **Stage 10 Hyperparameter Tuning:** `TimeSeriesSplit` cross-validation with `GridSearchCV` / `RandomizedSearchCV`.
4. **Evaluation:** Testing across 2023 all-season chronological test set and 7 Unseen Stations.
5. **Default vs. Tuned Comparison Table.**
6. **Stage 11 / Task 7 Explainability & Error Analysis:** Permutation Importance, Partial Dependence Plots (PDP), and Top 3 Failure Cases.

In [ ]:
import sys
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features import build_b2_features
from src.models.mlp import (
    build_mlp_regressor,
    build_mlp_classifier,
    prepare_mlp_data,
    get_clean_features,
    run_mlp_experiment,
    tune_mlp_regressor,
    threshold_health_score,
    plot_permutation_importance,
    plot_partial_dependence,
    get_top_failure_cases,
)
from src.evaluate import split_chronological, split_unseen_stations, evaluate_regression, evaluate_classification
print("Setup completed successfully.")

## 1. Load Real Dataset and Perform Chronological Splits

In [ ]:
parquet_path = ROOT / "data" / "processed" / "hkust_15min_v1.parquet"
df = pd.read_parquet(parquet_path)
print(f"Loaded dataset with {len(df):,} total rows across all inverters.")

# 1. Chronological split (2021-mid 2022 train, mid-end 2022 val, 2023 test)
train_df, val_df, test_df = split_chronological(df)

# 2. Unseen station holdout split (7 held-out rooftops)
_, unseen_df = split_unseen_stations(df)

## 2. Train and Evaluate Baseline MLP Models
- `mlp_regressor`: Continuous Health Score (0–100)
- `mlp_classifier`: Operational Status (`Healthy`, `Warning`, `Faulty`)
- Method 1 vs. Method 2 comparison on `Faulty` Recall.

In [ ]:
mlp_regressor, mlp_classifier = run_mlp_experiment(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    unseen_df=unseen_df,
)

## 3. Loss Curves: Verifying Neural Network Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Regressor loss curve
reg_losses = mlp_regressor.named_steps["mlp"].loss_curve_
axes[0].plot(reg_losses, color="#1f77b4", lw=2, marker="o", markersize=3)
axes[0].set_title("MLP Regressor Loss Progression (MSE)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch / Iteration")
axes[0].set_ylabel("Training Loss")
axes[0].grid(True, alpha=0.3)

# Classifier loss curve
clf_losses = mlp_classifier.named_steps["mlp"].loss_curve_
axes[1].plot(clf_losses, color="#2ca02c", lw=2, marker="s", markersize=3)
axes[1].set_title("MLP Classifier Loss Progression (Log-Loss)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch / Iteration")
axes[1].set_ylabel("Cross-Entropy Loss")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Stage 10: Hyperparameter Tuning with TimeSeriesSplit
- 3-fold expanding window cross-validation.
- Tune on 15% sample (~78k rows) -> refit winning architecture on full 523k rows.

In [ ]:
train_prep = prepare_mlp_data(train_df)
X_train, y_reg_train, _, _ = get_clean_features(train_prep)
best_params = tune_mlp_regressor(X_train, y_reg_train, n_iter=8, sample_fraction=0.15)
print("Optimal Hyperparameters:", best_params)

## 5. Master Experiment Log (`results/results.csv`)

In [ ]:
results_file = ROOT / "results" / "results.csv"
if results_file.exists():
    res_df = pd.read_csv(results_file)
    b2_results = res_df[res_df["student"] == "B2"]
    print(b2_results.to_string(index=False))
else:
    print("results/results.csv not found.")

## 6. Stage 11 / Task 7: Model Explainability & Error Analysis

Evaluators and operations engineers require transparent justification for neural network outputs:
1. **Permutation Feature Importance:** Measures sensitivity and degradation in RMSE when feature columns are scrambled.
2. **Partial Dependence Plots (PDP):** Maps physical response curves for irradiance, ambient temperature, and capacity.
3. **Top 3 Failure Case Diagnostics:** Evaluates the largest residual errors on the 2023 test set.

In [ ]:
# 1. Permutation Feature Importance
test_prep = prepare_mlp_data(test_df)
X_test, y_reg_test, _, features = get_clean_features(test_prep)

perm_result = plot_permutation_importance(
    model=mlp_regressor,
    X=X_test,
    y=y_reg_test,
    feature_names=features,
    save_path=str(ROOT / "reports" / "figures" / "b2_permutation_importance.png"),
)

In [ ]:
# 2. Partial Dependence Plots (Physical Response Curves)
train_prep = prepare_mlp_data(train_df)
X_train, _, _, _ = get_clean_features(train_prep)

plot_partial_dependence(
    model=mlp_regressor,
    X=X_train,
    features_to_plot=["irradiance_wm2", "temp_c", "capacity_kwp"],
    save_path=str(ROOT / "reports" / "figures" / "b2_partial_dependence.png"),
)

In [ ]:
# 3. Top 3 Worst Failure Cases on Test Set
failures_df = get_top_failure_cases(
    model=mlp_regressor,
    test_df=test_prep,
    X_test=X_test,
    y_test=y_reg_test,
    top_n=3,
)

### Physical Domain Analysis of Top 3 Failure Cases:
- **Observation:** All 3 worst errors occurred at `zone_d_inverter_2` during sunset hours (17:30 to 17:45 in October/November) when solar irradiance fell below $6 \text{ W/m}^2$.
- **Physical Mechanism:** At dusk, solar modules produce near-zero power ($95\text{--}157\text{ W}$) on a $50\text{ kW}$ array. Under normal daylight conditions, producing $<200\text{ W}$ indicates severe equipment failure (causing the model to predict a low health score of ~10–22 points). However, at twilight, this is normal inverter shutdown behaviour.
- **Engineering Mitigation:** In the operational dashboard, an irradiance threshold mask (`irradiance_wm2 >= 10 W/m²`) should gate the health evaluation to prevent dawn/dusk false alarms.